# J2C-TST-EN: dataset analysis

Paper-oriented analysis of the official English corpus. The notebook checks data integrity, describes splits and platforms, quantifies source-to-target stylistic transformations, and exports reusable tables and publication-quality figures.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy import stats

LANGUAGE = "EN"
RANDOM_SEED = 42
SPLIT_ORDER = ["train", "validation", "test"]

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "4-official_corpora").is_dir() and (candidate / "6-evaluation").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Cannot locate the J2C-TST project root")

PROJECT_ROOT = find_project_root()
CORPUS_PATH = PROJECT_ROOT / "4-official_corpora" / LANGUAGE / f"J2C_TST_{LANGUAGE}.csv"
NOTEBOOK_DIR = PROJECT_ROOT / "7-datasets_analysis" / f"J2C-TST-{LANGUAGE}"
ARTIFACTS_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(context="paper", style="whitegrid", palette="colorblind")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.titleweight": "bold",
    "font.size": 10,
})

print(f"Project root: {PROJECT_ROOT}")
print(f"Input corpus: {CORPUS_PATH}")
print(f"Artifacts: {ARTIFACTS_DIR}")

## 1. Load the official corpus

In [ ]:
df = pd.read_csv(CORPUS_PATH)
required = {
    "journalistic_style", "conversational_style", "social_network", "split",
    "title", "author", "license",
}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
if df.empty:
    raise ValueError("The official corpus is empty")

df["split"] = pd.Categorical(df["split"], categories=SPLIT_ORDER, ordered=True)
if df["split"].isna().any():
    raise ValueError("Unexpected or missing split labels")
if df["social_network"].isna().any():
    raise ValueError("Missing social-network labels")

print(f"Loaded {len(df):,} aligned pairs from {CORPUS_PATH.name}")

## 2. Dataset composition

In [ ]:
split_counts = (
    df.groupby("split", observed=False).size().rename("n").reset_index()
)
split_counts["percent"] = 100 * split_counts["n"] / len(df)

platform_split = pd.crosstab(df["social_network"], df["split"], margins=True)
platform_split.to_csv(ARTIFACTS_DIR / "table_02_platform_by_split.csv")
display(split_counts.round(2), platform_split)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sns.barplot(data=split_counts, x="split", y="n", ax=axes[0])
axes[0].set(title="Examples by split", xlabel="", ylabel="Examples")

platform_pct = pd.crosstab(df["social_network"], df["split"], normalize="columns") * 100
platform_pct.T.plot(kind="bar", stacked=True, ax=axes[1], width=0.75)
axes[1].set(title="Platform composition within each split", xlabel="", ylabel="Percent")
axes[1].legend(title="Platform", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "figure_01_dataset_composition.png")
plt.show()
plt.close(fig)

## 3. Text and social-style features

The feature set mirrors the editorial constraints used in evaluation and adds interpretable length and social-marker statistics.

In [ ]:
WORD_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)
SENTENCE_RE = re.compile(r"[.!?]+")
EMOJI_RE = re.compile(
    "[\U0001F1E6-\U0001F1FF\U0001F300-\U0001FAFF\u2600-\u27BF]",
    flags=re.UNICODE,
)

def text_features(series, prefix):
    text = series.fillna("").astype(str)
    out = pd.DataFrame(index=series.index)
    out[f"{prefix}_characters"] = text.str.len()
    out[f"{prefix}_words"] = text.map(lambda value: len(WORD_RE.findall(value)))
    out[f"{prefix}_sentences"] = text.map(lambda value: max(1, len(SENTENCE_RE.findall(value))))
    out[f"{prefix}_hashtags"] = text.str.count(r"(?<!\w)#\w+")
    out[f"{prefix}_mentions"] = text.str.count(r"(?<!\w)@\w+")
    out[f"{prefix}_emojis"] = text.map(lambda value: len(EMOJI_RE.findall(value)))
    out[f"{prefix}_has_url"] = text.str.contains(r"<URL>|https?://|www\.", case=False, regex=True)
    return out

features = pd.concat([
    df,
    text_features(df["journalistic_style"], "source"),
    text_features(df["conversational_style"], "target"),
], axis=1)
features["character_compression_ratio"] = (
    features["target_characters"] / features["source_characters"].replace(0, np.nan)
)
features["character_delta"] = features["target_characters"] - features["source_characters"]
features["gold_length_ok"] = features["target_characters"].between(100, 500)
features["gold_hashtags_ok"] = features["target_hashtags"] <= 3
features["gold_emojis_ok"] = features["target_emojis"] <= 3
features["gold_all_constraints_ok"] = features[[
    "gold_length_ok", "gold_hashtags_ok", "gold_emojis_ok"
]].all(axis=1)

numeric_features = [
    "source_characters", "target_characters", "source_words", "target_words",
    "target_hashtags", "target_mentions", "target_emojis", "character_compression_ratio",
]
summary = (
    features[numeric_features]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .T[["count", "mean", "std", "25%", "50%", "75%"]]
    .rename(columns={"25%": "q1", "50%": "median", "75%": "q3"})
)
summary.to_csv(ARTIFACTS_DIR / "table_03_text_features.csv")
display(summary.round(2))

## 4. Platform-level corpus profile

In [ ]:
platform_profile = (
    features.groupby("social_network", observed=True)
    .agg(
        n=("social_network", "size"),
        source_chars_median=("source_characters", "median"),
        target_chars_median=("target_characters", "median"),
        compression_median=("character_compression_ratio", "median"),
        hashtag_rate=("target_hashtags", lambda x: (x > 0).mean()),
        mention_rate=("target_mentions", lambda x: (x > 0).mean()),
        emoji_rate=("target_emojis", lambda x: (x > 0).mean()),
        url_rate=("target_has_url", "mean"),
        gold_constraints_rate=("gold_all_constraints_ok", "mean"),
    )
    .reset_index()
)
rate_columns = [c for c in platform_profile if c.endswith("_rate")]
platform_profile[rate_columns] *= 100
platform_profile.to_csv(ARTIFACTS_DIR / "table_04_platform_profile.csv", index=False)
display(platform_profile.round(2))

plot_data = features.melt(
    id_vars=["social_network"],
    value_vars=["source_characters", "target_characters"],
    var_name="text_role", value_name="characters",
)
plot_data["text_role"] = plot_data["text_role"].map({
    "source_characters": "Journalistic source",
    "target_characters": "Conversational target",
})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(
    data=plot_data, x="social_network", y="characters", hue="text_role",
    showfliers=False, ax=axes[0],
)
axes[0].set_yscale("log")
axes[0].set(title="Text length by platform", xlabel="Platform", ylabel="Characters (log scale)")
axes[0].legend(title="")

marker_plot = platform_profile.melt(
    id_vars="social_network",
    value_vars=["hashtag_rate", "mention_rate", "emoji_rate", "url_rate", "gold_constraints_rate"],
    var_name="feature", value_name="percent",
)
sns.barplot(data=marker_plot, x="social_network", y="percent", hue="feature", ax=axes[1])
axes[1].set(title="Gold target: social markers and compliance", xlabel="Platform", ylabel="Percent")
axes[1].legend(title="", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "figure_02_platform_profile.png")
plt.show()
plt.close(fig)

## 5. Statistical comparisons and effect sizes

In [ ]:
def cramers_v(table):
    chi2 = stats.chi2_contingency(table, correction=False).statistic
    n = table.to_numpy().sum()
    r, k = table.shape
    return float(np.sqrt((chi2 / n) / max(1, min(k - 1, r - 1))))

contingency = pd.crosstab(df["social_network"], df["split"])
chi = stats.chi2_contingency(contingency, correction=False)
tests = [{
    "comparison": "platform x split",
    "test": "chi-square",
    "statistic": chi.statistic,
    "p_value": chi.pvalue,
    "effect": cramers_v(contingency),
    "effect_name": "Cramer's V",
}]

platforms = sorted(features["social_network"].dropna().unique())
if len(platforms) == 2:
    a = features.loc[features["social_network"] == platforms[0], "target_characters"].to_numpy()
    b = features.loc[features["social_network"] == platforms[1], "target_characters"].to_numpy()
    mw = stats.mannwhitneyu(a, b, alternative="two-sided")
    rank_biserial = 2 * mw.statistic / (len(a) * len(b)) - 1
    tests.append({
        "comparison": f"target length: {platforms[0]} vs {platforms[1]}",
        "test": "Mann-Whitney U",
        "statistic": mw.statistic,
        "p_value": mw.pvalue,
        "effect": rank_biserial,
        "effect_name": "rank-biserial r",
    })

paired = stats.wilcoxon(
    features["source_characters"], features["target_characters"],
    alternative="two-sided", zero_method="wilcox",
)
tests.append({
    "comparison": "source vs target characters (paired)",
    "test": "Wilcoxon signed-rank",
    "statistic": paired.statistic,
    "p_value": paired.pvalue,
    "effect": float(features["character_delta"].median()),
    "effect_name": "median paired delta",
})

tests = pd.DataFrame(tests)
tests.to_csv(ARTIFACTS_DIR / "table_05_statistical_tests.csv", index=False)
display(tests)

## 6. Paper-ready findings

In [ ]:
largest_platform = platform_profile.sort_values("n", ascending=False).iloc[0]
smallest_platform = platform_profile.sort_values("n", ascending=True).iloc[0]
overall_compression = features["character_compression_ratio"].median()
overall_compliance = 100 * features["gold_all_constraints_ok"].mean()

if LANGUAGE == "EN":
    findings = [
        f"The official corpus contains **{len(df):,} aligned pairs** across {df['social_network'].nunique()} platforms.",
        f"The largest platform is **{largest_platform['social_network']}** ({int(largest_platform['n']):,} examples); the smallest is **{smallest_platform['social_network']}** ({int(smallest_platform['n']):,}).",
        f"Conversational targets retain a median of **{overall_compression:.1%}** of source characters, showing the magnitude of compression induced by the task.",
        f"**{overall_compliance:.1f}%** of gold targets satisfy the evaluation's combined length, hashtag, and emoji constraints.",
        "Platform comparisons are descriptive: content and source distributions may confound apparent platform effects.",
    ]
else:
    findings = [
        f"The official corpus contains **{len(df):,} aligned pairs** across {df['social_network'].nunique()} platforms.",
        f"The largest platform is **{largest_platform['social_network']}** ({int(largest_platform['n']):,} examples); the smallest is **{smallest_platform['social_network']}** ({int(smallest_platform['n']):,}).",
        f"Conversational targets retain a median of **{overall_compression:.1%}** of source characters, showing the magnitude of compression induced by the task.",
        f"**{overall_compliance:.1f}%** of gold targets satisfy the evaluation's combined length, hashtag, and emoji constraints.",
        "Platform comparisons are descriptive: content and source distributions may confound apparent platform effects.",
    ]

display(Markdown("\n".join(f"- {item}" for item in findings)))

summary_payload = {
    "language": LANGUAGE,
    "rows": int(len(df)),
    "split_counts": {str(k): int(v) for k, v in df["split"].value_counts(sort=False).items()},
    "platform_counts": {str(k): int(v) for k, v in df["social_network"].value_counts().items()},
    "median_character_compression_ratio": float(overall_compression),
    "gold_all_constraints_rate": float(features["gold_all_constraints_ok"].mean()),
}
(ARTIFACTS_DIR / "dataset_summary.json").write_text(
    json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"Saved paper tables, figures, and summary to {ARTIFACTS_DIR}")